# CMSC 25750 HW1 — LLM Interpretability

This notebook is your **only** submission. Run every cell top-to-bottom, keep all outputs visible, **export this file as PDF** and upload to Gradescope.

### What goes where

| Section | You implement | This notebook |
|---------|--------------|---------------|
| 1 Probing | `src/probing.py` | loads/trains, prints results |
| 2 Logit Lens | `src/logit_lens.py` | loads/computes, renders plots |
| 3 Attention Mechanics | — (no code) | written answers only |
| 4 KV Cache | `src/kv_cache.py` | loads/benchmarks, prints results |

**Do not write implementation code here.** Your only edits are filling in the `**Your answer:**` markdown cells.

Heavy computations (probing, logit lens, benchmark) are cached in `cache/` after the first run — subsequent runs load instantly.

In [ ]:
!uv pip install --system datasets transformers nnsight matplotlib torch psutil 2>/dev/null || \
 pip install datasets transformers nnsight matplotlib torch psutil -q

In [ ]:
import sys, os, json, pathlib
HW1_DIR = os.path.abspath('')
if HW1_DIR not in sys.path:
    sys.path.insert(0, HW1_DIR)

import torch, numpy as np, matplotlib.pyplot as plt
from tqdm import tqdm

device = (torch.device('cuda')  if torch.cuda.is_available()  else
          torch.device('mps')   if torch.backends.mps.is_available() else
          torch.device('cpu'))
print(f'device: {device}')

CACHE_DIR = pathlib.Path(HW1_DIR) / 'cache'
CACHE_DIR.mkdir(exist_ok=True)

---
### Full Autograder *(optional — run anytime for a complete check)*
Runs all tests across all sections. You can also run section-specific checks at the start of each section.

In [ ]:
import subprocess
result = subprocess.run(
    [sys.executable, '-m', 'pytest', 'tests/', '-v', '--tb=short', '--no-header'],
    capture_output=True, text=True, cwd=HW1_DIR,
)
print(result.stdout)
if result.returncode != 0:
    print('⚠  Some tests failed.')
else:
    print('✓  All tests passed.')

---
## Section 1 — Linear Probing (20 pts)

See **hw1.pdf Section 1** for background, task description, and probe specification.

Implement **Q1.1–Q1.3** in `src/probing.py`, run the autograder above, then run the cell below.

In [ ]:
# Section 1 autograder — run after implementing Q1.1–Q1.3 in src/probing.py
result = subprocess.run(
    [sys.executable, '-m', 'pytest', 'tests/test_probing.py', '-v', '--tb=short', '--no-header'],
    capture_output=True, text=True, cwd=HW1_DIR,
)
print(result.stdout)
print('✓ Section 1 tests passed.' if result.returncode == 0 else '⚠  Some Section 1 tests failed.')

**Before running this cell**: Compile Section 1 to generate cache files.

```bash
srun --partition general --gres gpu:a40:1 --time 0:20:00 python compile.py section1
```

The cell below will load results from `cache/probing_results.json`.

In [ ]:
from src.probing import (get_model_and_tokenizer, get_data, build_classifier,
                         get_sentence_repr, train_probe, evaluate_probe)
from tqdm import tqdm

PROBING_CACHE = CACHE_DIR / 'probing_results.json'

if PROBING_CACHE.exists():
    with open(PROBING_CACHE) as f:
        probing_results = json.load(f)
    print('✓ Loaded probing results from cache — skipping training.')
else:
    torch.manual_seed(42)
    train_data, test_data = get_data(seed=42)
    model_p, tokenizer_p, emb_dim = get_model_and_tokenizer('EleutherAI/pythia-160m', device)

    print('Extracting representations (may take a few minutes on CPU)…')
    train_repr = [get_sentence_repr(s, model_p, tokenizer_p, device) 
                  for s in tqdm(train_data['sentence'], desc='Train set')]
    test_repr  = [get_sentence_repr(s, model_p, tokenizer_p, device) 
                  for s in tqdm(test_data['sentence'], desc='Test set')]

    n_layers             = train_repr[0].shape[0]
    final_idx, third_idx = n_layers - 1, 3

    def mean_pool(lst, idx):
        return [torch.tensor(np.mean(r[idx], 0), dtype=torch.float32).to(device) for r in lst]

    lbl_tr = [torch.tensor(int(x), dtype=torch.long).to(device) for x in train_data['label']]
    lbl_te = [torch.tensor(int(x), dtype=torch.long).to(device) for x in test_data['label']]

    print('\nTraining final-layer probe…')
    clf_f, crit_f, opt_f = build_classifier(emb_dim, 2, str(device))
    trL_f, trA_f = train_probe(mean_pool(train_repr, final_idx), lbl_tr, clf_f, crit_f, opt_f)
    teL_f, teA_f = evaluate_probe(mean_pool(test_repr, final_idx), lbl_te, clf_f, crit_f)

    print('\nTraining third-layer probe…')
    clf_t, crit_t, opt_t = build_classifier(emb_dim, 2, str(device))
    trL_t, trA_t = train_probe(mean_pool(train_repr, third_idx), lbl_tr, clf_t, crit_t, opt_t)
    teL_t, teA_t = evaluate_probe(mean_pool(test_repr, third_idx), lbl_te, clf_t, crit_t)

    probing_results = {
        'final_layer_idx': final_idx,
        'final': {'train_acc': trA_f, 'test_acc': teA_f, 'train_loss': trL_f, 'test_loss': teL_f},
        'third': {'train_acc': trA_t, 'test_acc': teA_t, 'train_loss': trL_t, 'test_loss': teL_t},
    }
    with open(PROBING_CACHE, 'w') as f:
        json.dump(probing_results, f, indent=2)
    print('\n✓ Saved to cache/probing_results.json')

In [ ]:
r = probing_results
L = r['final_layer_idx']
print(f"{'Layer':<20} {'Train Acc':>10} {'Test Acc':>10}")
print('-' * 42)
print(f"{'Final (L='+str(L)+')':<20} {r['final']['train_acc']:>10.4f} {r['final']['test_acc']:>10.4f}")
print(f"{'Third  (L=3)':<20} {r['third']['train_acc']:>10.4f} {r['third']['test_acc']:>10.4f}")
gap = r['final']['test_acc'] - r['third']['test_acc']
print(f"\nTest accuracy gap (final − third): {gap:+.4f} ({gap*100:+.2f} pp)")

### 1.4(a) — Layer comparison [4 pts]

State the test accuracy for both layers and the gap between them. What does this suggest about where syntactic information is stored in a transformer?

**Your answer:** *(replace this text)*

### 1.4(b) — Linear vs. non-linear [4 pts]

Why is a linear probe preferred over a more powerful classifier (e.g. an MLP) when the goal is to interpret model representations?

**Your answer:** *(replace this text)*

### 1.4(c) — Null experiment [6 pts]

Suppose you train the same linear probe on hidden states from a **randomly initialized** (untrained) model of the same architecture and obtain high test accuracy. Can you conclude whether the trained model's representations encode subject-verb agreement? Why or why not?

**Your answer:** *(replace this text)*

---
## Section 2 — Logit Lens (16 pts)

See **hw1.pdf Section 2** for background on the logit lens and the residual stream. Read [Olsson et al. (2022)](https://arxiv.org/abs/2209.11895) before answering the writing questions.

Implement **Q2.1–Q2.2** in `src/logit_lens.py`, then run the cell below.

** Before running this cell**: Compile Section 2 to generate cache files.

```bash
srun --partition general --gres gpu:a40:1 --time 0:15:00 python compile.py section2
```

The cell below will load results from `cache/logit_lens_results.pt`.

In [ ]:
from src.logit_lens import get_logit_lens_predictions, get_token_rank_by_layer

LOGIT_CACHE = CACHE_DIR / 'logit_lens_results.pt'
PROMPT_A    = 'The currency in the United States of America is the dollar.'
PROMPT_B    = 'The result of two plus two is four.'

def encode_single(s, tok):
    for cand in [s, ' ' + s]:
        ids = tok.encode(cand)
        if len(ids) == 1: return ids[0]
    return tok.encode(' ' + s)[-1]

if LOGIT_CACHE.exists():
    data = torch.load(LOGIT_CACHE, map_location='cpu', weights_only=False)
    mp_a, tt_a, iw_a   = data['mp_a'], data['tt_a'], data['iw_a']
    mp_b, tt_b, iw_b   = data['mp_b'], data['tt_b'], data['iw_b']
    ranks_a, ranks_b   = data['ranks_a'], data['ranks_b']
    id_dollar, id_four = data['id_dollar'], data['id_four']
    print('✓ Loaded logit lens results from cache — skipping nnsight computation.')
else:
    from nnsight import LanguageModel
    lm = LanguageModel('EleutherAI/pythia-410m', device_map='auto', dispatch=True)

    mp_a, tt_a, iw_a, probs_a = get_logit_lens_predictions(PROMPT_A, lm)
    mp_b, tt_b, iw_b, probs_b = get_logit_lens_predictions(PROMPT_B, lm)

    id_dollar = encode_single('dollar', lm.tokenizer)
    id_four   = encode_single('four',   lm.tokenizer)
    ranks_a   = get_token_rank_by_layer(probs_a, id_dollar)
    ranks_b   = get_token_rank_by_layer(probs_b, id_four)

    torch.save({
        'mp_a': mp_a.cpu(), 'tt_a': tt_a, 'iw_a': iw_a,
        'mp_b': mp_b.cpu(), 'tt_b': tt_b, 'iw_b': iw_b,
        'ranks_a': ranks_a, 'ranks_b': ranks_b,
        'id_dollar': id_dollar, 'id_four': id_four,
    }, LOGIT_CACHE)
    print('✓ Saved logit lens results to cache/logit_lens_results.pt')

In [ ]:
def plot_logit_lens(max_probs, top_tokens, input_words, title='Logit Lens'):
    """Heatmap of max-probability token per layer/position. (Given)"""
    data = max_probs.detach().cpu().numpy()
    fig, ax = plt.subplots(figsize=(max(10, len(input_words) * 0.9), 6))
    im = ax.imshow(data, aspect='auto', cmap='RdYlBu_r', vmin=0, vmax=1)
    ax.set_xticks(range(len(input_words)))
    ax.set_xticklabels(input_words, rotation=60, ha='right', fontsize=9)
    ax.set_yticks(range(len(top_tokens)))
    ax.set_yticklabels(range(len(top_tokens)))
    ax.set_xlabel('Input tokens'); ax.set_ylabel('Layer'); ax.set_title(title)
    for i in range(data.shape[0]):
        for j in range(data.shape[1]):
            ax.text(j, i, top_tokens[i][j], ha='center', va='center', fontsize=6)
    fig.colorbar(im, ax=ax, label='Max prob')
    ax.invert_yaxis(); plt.tight_layout(); plt.show()

In [ ]:
plot_logit_lens(mp_a, tt_a, iw_a, '(a) Commonsense')
plot_logit_lens(mp_b, tt_b, iw_b, '(b) Math')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, ranks, label, tgt in [
    (axes[0], ranks_a, '(a) Commonsense', '"dollar"'),
    (axes[1], ranks_b, '(b) Math',        '"four"'),
]:
    ax.plot(ranks, marker='o', ms=4)
    ax.set_title(f'{label}\nRank of {tgt} by layer')
    ax.set_xlabel('Layer'); ax.set_ylabel('Rank (1 = top)')
    ax.invert_yaxis(); ax.set_yscale('log'); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

first_a = next((i for i, r in enumerate(ranks_a) if r == 1), None)
first_b = next((i for i, r in enumerate(ranks_b) if r == 1), None)
print(f'"dollar" first reaches rank 1 at layer: {first_a}')
print(f'"four"   first reaches rank 1 at layer: {first_b}')
print(f'Difference (commonsense − math):         {(first_a or 0) - (first_b or 0):+d} layers')

### 2.3(a) [3 pts]

Suppose you observe the correct token reaching rank 1 at some layer $\ell^* < L$. Is it correct to say the model "decided" the answer at $\ell^*$?

**Your answer:** *(replace this text)*

### 2.3(b) [3 pts]

According to [Olsson et al. (2022)](https://arxiv.org/abs/2209.11895), explain why in-context examples cause the correct token to rank highly at earlier layers.

**Your answer:** *(replace this text)*

---
## Section 3 — Attention Mechanics (20 pts)

Show every intermediate step. Jupyter renders LaTeX math in markdown cells. See hw1.pdf Section 3 for full problem statements.*

### 3.1 — QKV Attention by Hand [8 pts]

Autoregressive decoder-only model, $T=3$ tokens, $d_k=3$:

$$Q = \begin{bmatrix} 1 & 0 & 1 \\ 0 & 1 & 0 \\ 1 & 1 & 1 \end{bmatrix}, \quad
K = \begin{bmatrix} 1 & 1 & 0 \\ 0 & 1 & 1 \\ 1 & 0 & 1 \end{bmatrix}, \quad
V = \begin{bmatrix} 2 & 0 & 1 \\ 1 & 1 & 0 \\ 0 & 1 & 2 \end{bmatrix}$$

Compute $\operatorname{Attention}(Q,K,V) = \operatorname{softmax}(QK^\top/\sqrt{d_k})\,V$ step by step, applying the autoregressive constraint. Show all four intermediate matrices.

**Your answer:** *(replace this text)*

### 3.2 — FLOPs in Attention [6 pts]

1 FMA = 2 FLOPs. An $m{\times}n$ by $n{\times}p$ multiply costs $2mnp$ FLOPs. Count only matrix-multiply FLOPs; ignore softmax.

| Symbol | Value |
|--------|-------|
| $d$ | 512 | $T$ | 256 | $L$ | 6 |
| $W_Q,W_K,W_V,W_O$ | $\in\mathbb{R}^{d\times d}$ |

**(a) [4 pts]** Derive total FLOPs for one forward pass of single-head attention with batch $B$, sequence $T$, dimension $d$ (all six components: $Q$, $K$, $V$ projections, scores $QK^\top$, weighted sum $AV$, output $W_O$). Sum to one formula, then evaluate for $B=4$.

**(b) [2 pts]** At what $T$ does the $T^2$ term dominate? Express in terms of $d$, evaluate for $d=512$.

**3.2(a):** *(your answer)*

**3.2(b):** *(your answer)*

### 3.3 — KV-Cache: Memory and FLOPs [6 pts]

Use $B=4$, $d=512$, $L=6$, float16 ($b=2$ bytes/element).

**(a) [3 pts]** Derive the formula for KV-cache size in bytes (one $K$ and one $V$ per layer per token, across all $L$ layers, $B$ sequences, $T_\text{prev}$ tokens, hidden dim $d$). Evaluate for $T_\text{prev}=256$.

**(b) [3 pts]** Write the FLOPs for one decode step **without** cache (re-run full sequence of length $T_\text{prev}+1$, using your 3.2(a) formula with $B=4$) and **with** cache (process only the new token against $T_\text{prev}$ cached keys). Show the per-step speedup simplifies to $T_\text{prev}$; evaluate for $T_\text{prev}=256$.

**3.3(a):** *(your answer)*

**3.3(b):** *(your answer)*

---
## Section 4 — Multi-Layer KV Cache (17 pts)

See **hw1.pdf Section 4** for the problem setup and motivation.

Implement **Q4.1–Q4.4** in `src/kv_cache.py` (`MultiLayerKVCache`), run the autograder above, then run the cells below.

**Note**: The benchmark cell uses `ranks_a` from Section 2. Make sure Section 2 has already run before running Section 4.

In [ ]:
# Section 4 autograder — run after implementing Q4.1–Q4.4 in src/kv_cache.py
result = subprocess.run(
    [sys.executable, '-m', 'pytest', 'tests/test_kv_cache.py', '-v', '--tb=short', '--no-header'],
    capture_output=True, text=True, cwd=HW1_DIR,
)
print(result.stdout)
print('✓ Section 4 tests passed.' if result.returncode == 0 else '⚠  Some Section 4 tests failed.')

In [ ]:
from src.kv_cache import MultiLayerKVCache, generate_reference, generate_with_budget, benchmark
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_NAME = 'EleutherAI/pythia-160m'
model_kv     = AutoModelForCausalLM.from_pretrained(MODEL_NAME).to(device)
tokenizer_kv = AutoTokenizer.from_pretrained(MODEL_NAME)
model_kv.eval()

L         = model_kv.config.num_hidden_layers       # 12
num_heads = getattr(model_kv.config, 'num_key_value_heads', model_kv.config.num_attention_heads)     # 12
d_head    = model_kv.config.hidden_size // model_kv.config.num_attention_heads  # 64

# Full-cache token capacity (per-layer budget when budget = 100%)
FULL_BUDGET = L * 256   # 12 layers × 256 tokens each = 3072 total tokens

print(f'Model: {MODEL_NAME}  |  L={L}, H={num_heads}, d_h={d_head}')
print(f'Full budget: {FULL_BUDGET} tokens across {L} layers')

In [ ]:
GEN_PROMPT   = ('The transformer architecture was introduced in "Attention is All You Need". '
                'It uses self-attention to process sequences in parallel. Since then,')
MAX_NEW_TOKS = 50

# Reference: unlimited cache (ground truth for quality comparison)
ref_out = generate_reference(model_kv, tokenizer_kv, GEN_PROMPT, MAX_NEW_TOKS)
print('Reference output:')
print(ref_out)

In [ ]:
# Define budget levels and layer importance scores for KV cache experiments

# Budget levels to test (25%, 50%, 75%, 100% of full capacity)
budgets = [
    FULL_BUDGET // 4,      # 768 tokens (25%)
    FULL_BUDGET // 2,      # 1536 tokens (50%)
    FULL_BUDGET * 3 // 4,  # 2304 tokens (75%)
    FULL_BUDGET            # 3072 tokens (100%)
]

# Layer importance scores from Section 2 logit lens results
# Note: Section 2 uses pythia-410m (24 layers) but Section 4 uses pythia-160m (12 layers)
# We take the first L layers from ranks_a to match the current model
layer_scores = ranks_a[:L] if len(ranks_a) >= L else ranks_a

print(f"Budget levels: {budgets}")
print(f"Layer scores (first {L} from Section 2): {layer_scores}")
print(f"Model has {L} layers, using {len(layer_scores)} layer scores")

** Before running this cell**: Compile Section 4 to generate cache files.

```bash
srun --partition general --gres gpu:a40:1 --time 0:20:00 python compile.py section4
```

The cell below will load results from `cache/kv_cache_benchmark.json`.

Note: Section 4 uses `ranks_a` from Section 2, so make sure you've compiled Section 2 first.

In [ ]:
KV_BENCH_CACHE = CACHE_DIR / 'kv_cache_benchmark.json'

if KV_BENCH_CACHE.exists():
    with open(KV_BENCH_CACHE) as f:
        kv_bench = json.load(f)
    print('✓ Loaded benchmark results from cache.')
else:
    raw = benchmark(
        model_kv, tokenizer_kv, GEN_PROMPT,
        total_budgets=budgets,
        layer_scores=layer_scores,
        max_new_tokens=MAX_NEW_TOKS,
    )
    # Convert int keys → str for JSON serialisation
    kv_bench = {str(b): v for b, v in raw.items()}
    with open(KV_BENCH_CACHE, 'w') as f:
        json.dump(kv_bench, f, indent=2)
    print('✓ Saved to cache/kv_cache_benchmark.json')

In [ ]:
pct_labels  = [int(int(b) / FULL_BUDGET * 100) for b in kv_bench]
uni_evict   = [sum(kv_bench[b]['uniform_stats']['eviction_counts'])   for b in kv_bench]
pri_evict   = [sum(kv_bench[b]['priority_stats']['eviction_counts'])  for b in kv_bench]

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Left: evictions vs. budget
axes[0].plot(pct_labels, uni_evict, 'o-',  label='uniform',  color='steelblue')
axes[0].plot(pct_labels, pri_evict, 's--', label='priority', color='darkorange')
axes[0].set_xlabel('Budget (%)'); axes[0].set_ylabel('Total evictions')
axes[0].set_title('Evictions vs. budget')
axes[0].legend(); axes[0].grid(alpha=0.3)

# Right: per-layer budget at 50%
b50    = str(budgets[1])   # budgets = [25%, 50%, 75%, 100%] → index 1 is 50%
uni_pb = kv_bench[b50]['uniform_stats']['per_layer_budget']
pri_pb = kv_bench[b50]['priority_stats']['per_layer_budget']
x = range(len(uni_pb))

axes[1].bar([i - 0.2 for i in x], uni_pb, 0.38, label='uniform',  color='steelblue', alpha=0.8)
axes[1].bar([i + 0.2 for i in x], pri_pb, 0.38, label='priority', color='darkorange', alpha=0.8)
axes[1].set_xlabel('Layer index'); axes[1].set_ylabel('Token budget')
axes[1].set_title('Per-layer budget at 50% — uniform vs. priority')
axes[1].legend(); axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout(); plt.show()


### 4.5 — Analysis [5 pts]

**(a) [3 pts]** The priority policy gives more cache budget to layers where the target token has a *higher rank* (layers that haven't converged). What assumption does this make about the relationship between a layer's prediction uncertainty and its need for long-range context? Is this assumption always valid? Describe a scenario where it could fail even with a generous total budget.

**(b) [2 pts]** FIFO always evicts the *oldest* tokens first. Propose one alternative eviction criterion, explain what information it uses, and argue why it could preserve generation quality better than FIFO for long-context generation.

**Your answer (4.5a):** *(replace this text)*

**Your answer (4.5b):** *(replace this text)*